# Parameter sweep : Gaussian vs Logistic CMM

Figure D.2-style grid comparing Gaussian (noise) vs Logistic (FLXMRglm) drivers on mixed binary/continuous data.
Loads `results/sweep_20260430_1555/results.csv`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[2]))

import pandas as pd
from experiments.mixed_cmm.synthetic.config import PARAM_LABELS, DEFAULTS
from src_tb.causal_recovery.visualization import plot_sweep

RESULTS = Path.cwd().parents[2] / 'results' / 'sweep_20260430_1555' / 'results.csv'
df = pd.read_csv(RESULTS)
df.head()

In [ ]:
model_labels = {'gaussian': 'Gaussian (noise)', 'logistic': 'Logistic (FLXMRglm)'}
models_order = ['gaussian', 'logistic']

## Overall graph metrics (Figure D.2 style)

In [ ]:
metric_labels = {
    'sd':  'SD',
    'sc':  'S/C',
    'shd': 'SHD',
    'fpr': 'FPR-dir',
    'tpr': 'TPR-dir',
    'f1':  'F1-dir',
}
plot_sweep(df, metrics=list(metric_labels), metric_labels=metric_labels,
           param_labels=PARAM_LABELS, group_col='model', group_labels=model_labels,
           title='Gaussian vs Logistic CMM on mixed binary/continuous data')

## Edge-type breakdown

Splits F1/precision/recall by edge class: mutation→mutation (`bin_bin`) vs mutation→MIC (`bin_cont`).

In [ ]:
edge_metric_labels = {
    'f1_bin_bin':         'F1 (mut→mut)',
    'f1_bin_cont':        'F1 (mut→Y)',
    'precision_bin_bin':  'Prec (mut→mut)',
    'precision_bin_cont': 'Prec (mut→Y)',
    'recall_bin_bin':     'Rec (mut→mut)',
    'recall_bin_cont':    'Rec (mut→Y)',
}
plot_sweep(df, metrics=list(edge_metric_labels), metric_labels=edge_metric_labels,
           param_labels=PARAM_LABELS, group_col='model', group_labels=model_labels,
           title='Recovery by edge type')

## Defaults-only summary table

In [ ]:
param_to_default = {k: DEFAULTS[k] for k in ('n_mix', 'p_mix', 'n_samples', 'n_obs', 'p_graph')}
mask = df.apply(lambda r: r['value'] == param_to_default.get(r['param']), axis=1)
df_def = df[mask]
summary = df_def.groupby('model')[['f1', 'tpr', 'fpr', 'shd', 'f1_bin_bin', 'f1_bin_cont']].mean().round(3)
summary.loc[models_order]